# 10. Pairwise Sequence Alignment

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `nw.py`

Global alignment by the Needleman-Wunsch method, from scratch.


In [ ]:
"""Global alignment by the Needleman-Wunsch method, from scratch."""

MATCH, MISMATCH, GAP = 1, -1, -2


def score_pair(x: str, y: str) -> int:
    return MATCH if x == y else MISMATCH


def fill(a: str, b: str) -> list[list[int]]:
    """The scoring grid: grid[i][j] is the best score for a[:i], b[:j]."""
    grid = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        grid[i][0] = i * GAP
    for j in range(1, len(b) + 1):
        grid[0][j] = j * GAP
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            grid[i][j] = max(
                grid[i - 1][j - 1] + score_pair(a[i - 1], b[j - 1]),
                grid[i - 1][j] + GAP,  # a letter of a against a gap
                grid[i][j - 1] + GAP,  # a letter of b against a gap
            )
    return grid


def traceback(a: str, b: str, grid: list[list[int]]) -> tuple[str, str]:
    """Walk back from the bottom right corner to recover the alignment."""
    top, bottom = [], []
    i, j = len(a), len(b)
    while i > 0 or j > 0:
        step = score_pair(a[i - 1], b[j - 1]) if i and j else None
        if step is not None and grid[i][j] == grid[i - 1][j - 1] + step:
            top.append(a[i - 1])
            bottom.append(b[j - 1])
            i, j = i - 1, j - 1
        elif i > 0 and grid[i][j] == grid[i - 1][j] + GAP:
            top.append(a[i - 1])
            bottom.append("-")
            i -= 1
        else:
            top.append("-")
            bottom.append(b[j - 1])
            j -= 1
    return "".join(reversed(top)), "".join(reversed(bottom))


def mark(x: str, y: str) -> str:
    """| for a match, . for a mismatch, a space for a gap."""
    if "-" in (x, y):
        return " "
    return "|" if x == y else "."


def show(top: str, bottom: str) -> None:
    marks = "".join(mark(x, y) for x, y in zip(top, bottom))
    print(top, marks, bottom, sep="\n")


if __name__ == "__main__":
    a, b = "ATGAGCA", "AGACCA"
    grid = fill(a, b)
    for row in grid:
        print(" ".join(f"{value:3}" for value in row))
    print("Score:", grid[-1][-1])
    show(*traceback(a, b, grid))

## `peptide.py`

Ingram's sickle cell peptide, found in beta globin by local alignment.


In [ ]:
# Ingram's sickle cell peptide, found in beta globin by local alignment.
from Bio import Align, SeqIO
from Bio.Align import substitution_matrices

beta = next(SeqIO.parse("globins.fasta", "fasta")).seq
sickle_peptide = "VHLTPVEK"

aligner = Align.PairwiseAligner()
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

for mode in ("global", "local"):
    aligner.mode = mode
    best = aligner.align(beta, sickle_peptide)[0]
    print(f"{mode:6}  score {best.score:6.1f}")

print(best)

## `globins.py`

Human beta globin against mouse beta globin and human alpha globin. Sequences: UniProt P68871, P02088 and P69905 (CC BY 4.0).


In [ ]:
# Human beta globin against mouse beta globin and human alpha globin.
# Sequences: UniProt P68871, P02088 and P69905 (CC BY 4.0).
from Bio import Align, SeqIO
from Bio.Align import substitution_matrices

BLOSUM62 = substitution_matrices.load("BLOSUM62")


def mark(x: str, y: str) -> str:
    """| identical, : similar (scores above zero), . different."""
    if "-" in (x, y):
        return " "
    if x == y:
        return "|"
    return ":" if BLOSUM62[x][y] > 0 else "."


def show(alignment: Align.Alignment, width: int = 60) -> None:
    top, bottom = alignment[0], alignment[1]
    for start in range(0, len(top), width):
        a, b = top[start : start + width], bottom[start : start + width]
        print(a, "".join(mark(x, y) for x, y in zip(a, b)), b, sep="\n")
        print()


proteins = {}
for record in SeqIO.parse("globins.fasta", "fasta"):
    proteins[record.id.split("|")[2]] = record.seq

aligner = Align.PairwiseAligner(mode="global")
aligner.substitution_matrix = BLOSUM62
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

human = proteins["HBB_HUMAN"]
for name in ("HBB1_MOUSE", "HBA_HUMAN"):
    best = aligner.align(human, proteins[name])[0]
    counts = best.counts(BLOSUM62)
    print(name)
    print(f"  score      {best.score:6.1f}")
    print(f"  identical  {counts.identities:4} of {best.length}")
    print(f"  similar    {counts.positives:4} of {best.length}")
    print(f"  gaps       {counts.gaps:4}")
    print()

show(best)